# LeetCode #54 — Spiral Matrix: duyệt xoắn ốc bằng bốn biên

**Mức độ:** Medium · **Chủ đề:** Array, Matrix, Simulation

Skeleton trên LeetCode:

```python
class Solution:
    def spiralOrder(self, matrix: List[List[int]]) -> List[int]:
```

Mục tiêu của notebook là luyện **mô phỏng trên ma trận** với hai cách tiếp cận:

1. **Bốn biên** `top`, `bottom`, `left`, `right` co dần vào trong — gọn, `O(1)` bộ nhớ.
2. **Bảng hướng đi + đánh dấu đã thăm** — "đi thẳng, gặp tường thì rẽ phải".

Và quan trọng nhất: xử lý đúng các trường hợp **chỉ còn một hàng** hoặc **một cột** — nơi gần như mọi lỗi của bài này xảy ra.

# 1. Bài toán là gì?

Cho ma trận `m × n`, trả về mọi phần tử theo **thứ tự xoắn ốc**: bắt đầu từ góc trên trái, đi **phải → xuống → trái → lên**, rồi lặp lại vào trong.

**Constraints (theo đề):**

- `1 <= m, n <= 10`
- `-100 <= matrix[i][j] <= 100`

---

## Ví dụ 1

```text
1 2 3
4 5 6      ->  [1, 2, 3, 6, 9, 8, 7, 4, 5]
7 8 9
```

## Ví dụ 2

```text
1  2  3  4
5  6  7  8     ->  [1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]
9 10 11 12
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Ma trận (mảng hai chiều) trong Python

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Ma trận (mảng hai chiều) trong Python

### Tạo và truy cập

Ma trận `m × n` là list gồm `m` hàng, mỗi hàng là list `n` phần tử. Ô hàng `i`, cột `j` là `M[i][j]`; số hàng `len(M)`, số cột `len(M[0])`.

```text
        j=0 j=1 j=2
i=0   [  1,  2,  3 ]
i=1   [  4,  5,  6 ]
```

In [1]:
M_mx = [[1, 2, 3],
        [4, 5, 6]]
print("M[1][2] =", M_mx[1][2], "| số hàng:", len(M_mx), "| số cột:", len(M_mx[0]))

dung_mx = [[0] * 3 for _ in range(2)]    # ĐÚNG: mỗi hàng là list riêng
sai_mx = [[0] * 3] * 2                   # SAI: 2 hàng là cùng một list
dung_mx[0][0] = sai_mx[0][0] = 7
print("tạo bằng for:", dung_mx, "| nhân list:", sai_mx)

M[1][2] = 6 | số hàng: 2 | số cột: 3
tạo bằng for: [[7, 0, 0], [0, 0, 0]] | nhân list: [[7, 0, 0], [7, 0, 0]]


### Duyệt và các phép biến đổi cơ bản

- Duyệt theo hàng: `for i in range(m): for j in range(n)`.
- **Chuyển vị** (transpose): `T[j][i] = M[i][j]` — hàng thành cột. Trong Python: `list(zip(*M))`.
- **Lật ngang** một hàng: `row[::-1]`; lật dọc: đảo thứ tự hàng `M[::-1]`.
- Xoay 90° theo chiều kim đồng hồ = chuyển vị rồi lật từng hàng.

In [2]:
def in_mt(A):
    for row in A:
        print("   ", " ".join(f"{v:>2}" for v in row))

A_mx = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print("gốc:"); in_mt(A_mx)
T_mx = [list(r) for r in zip(*A_mx)]
print("chuyển vị:"); in_mt(T_mx)
print("xoay 90° phải (chuyển vị + lật hàng):"); in_mt([r[::-1] for r in T_mx])

gốc:
     1  2  3
     4  5  6
     7  8  9
chuyển vị:
     1  4  7
     2  5  8
     3  6  9
xoay 90° phải (chuyển vị + lật hàng):
     7  4  1
     8  5  2
     9  6  3


### Ô kề và mảng hướng

Bốn ô kề của `(r, c)`: lên `(r−1, c)`, xuống `(r+1, c)`, trái `(r, c−1)`, phải `(r, c+1)`. Viết thành **mảng hướng** để khỏi lặp code, và luôn **kiểm tra biên** `0 <= nr < m and 0 <= nc < n`.

In [3]:
HUONG = [(-1, 0), (1, 0), (0, -1), (0, 1)]       # lên, xuống, trái, phải
def o_ke(r, c, m, n):
    return [(r + dr, c + dc) for dr, dc in HUONG if 0 <= r + dr < m and 0 <= c + dc < n]
print("ô kề của (0, 0) trong lưới 3×3:", o_ke(0, 0, 3, 3))
print("ô kề của (1, 1):", o_ke(1, 1, 3, 3))

ô kề của (0, 0) trong lưới 3×3: [(1, 0), (0, 1)]
ô kề của (1, 1): [(0, 1), (2, 1), (1, 0), (1, 2)]


### Đánh số ô thành một chỉ số và ngược lại

Ma trận `m × n` "trải phẳng" theo hàng: ô `(i, j)` ↔ chỉ số `k = i * n + j`; ngược lại `i, j = divmod(k, n)`. Hữu ích để tìm nhị phân trên ma trận hoặc lưu ô vào set.

In [4]:
n_cols = 4
for k in [0, 5, 11]:
    print(f"k = {k:>2} → (i, j) = {divmod(k, n_cols)} → k = {divmod(k, n_cols)[0] * n_cols + divmod(k, n_cols)[1]}")

k =  0 → (i, j) = (0, 0) → k = 0
k =  5 → (i, j) = (1, 1) → k = 5
k = 11 → (i, j) = (2, 3) → k = 11


Lỗi hay gặp: `[[0]*n]*m`; nhầm `m` (hàng) với `n` (cột) khi ma trận không vuông; quên kiểm tra biên; sửa ma trận trong lúc vẫn cần giá trị cũ để tính ô khác.

## 2.2 Áp dụng vào bài này

Duyệt **ma trận** theo hình xoắn ốc bằng bốn biên `top, bottom, left, right` co dần sau mỗi cạnh, hoặc bằng **mảng hướng** (phải, xuống, trái, lên) và đổi hướng khi gặp biên/ô đã thăm.

# 3. Cách 1: bốn biên

Giữ bốn biên của phần **chưa duyệt**:

```text
top = 0, bottom = m - 1, left = 0, right = n - 1
```

Mỗi vòng "bóc" một lớp vỏ theo 4 bước:

1. Đi **phải** trên hàng `top`, từ `left` tới `right`. Xong thì `top += 1`.
2. Đi **xuống** trên cột `right`, từ `top` tới `bottom`. Xong thì `right -= 1`.
3. Nếu còn hàng (`top <= bottom`): đi **trái** trên hàng `bottom`, từ `right` về `left`. Xong thì `bottom -= 1`.
4. Nếu còn cột (`left <= right`): đi **lên** trên cột `left`, từ `bottom` về `top`. Xong thì `left += 1`.

Lặp khi `top <= bottom and left <= right`.

## Vì sao cần kiểm tra ở bước 3 và 4?

Sau bước 1 và 2, phần còn lại có thể đã **rỗng theo một chiều**. Ví dụ ma trận chỉ có **một hàng** `[[1, 2, 3]]`:

- Bước 1 đi `1, 2, 3`, `top` thành `1 > bottom = 0`.
- Nếu không kiểm tra, bước 3 sẽ đi **lại** hàng 0 từ phải sang trái → in thêm `2, 1`!

In [5]:
from typing import List

def spiral_order(matrix: List[List[int]]) -> List[int]:
    result = []
    top, bottom = 0, len(matrix) - 1
    left, right = 0, len(matrix[0]) - 1

    while top <= bottom and left <= right:
        for c in range(left, right + 1):           # 1. phải
            result.append(matrix[top][c])
        top += 1

        for r in range(top, bottom + 1):           # 2. xuống
            result.append(matrix[r][right])
        right -= 1

        if top <= bottom:
            for c in range(right, left - 1, -1):   # 3. trái
                result.append(matrix[bottom][c])
            bottom -= 1

        if left <= right:
            for r in range(bottom, top - 1, -1):   # 4. lên
                result.append(matrix[r][left])
            left += 1

    return result

print(spiral_order([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))
print(spiral_order([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]))
print(spiral_order([[1, 2, 3]]))
print(spiral_order([[1], [2], [3]]))

[1, 2, 3, 6, 9, 8, 7, 4, 5]
[1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]
[1, 2, 3]
[1, 2, 3]


`range(right, left - 1, -1)` đi **ngược** từ `right` về `left` (bao gồm cả `left`). Nhớ: `range(a, b, -1)` dừng **trước** `b`, nên phải viết `left - 1`.

# 4. Mô phỏng từng bước

In [6]:
def trace_spiral(matrix: List[List[int]]) -> List[int]:
    result = []
    top, bottom = 0, len(matrix) - 1
    left, right = 0, len(matrix[0]) - 1
    layer = 0
    while top <= bottom and left <= right:
        print(f"--- vòng {layer}: top={top} bottom={bottom} left={left} right={right}")
        seg = [matrix[top][c] for c in range(left, right + 1)]
        result += seg; top += 1
        print(f"  phải  (hàng {top - 1}): {seg}")
        seg = [matrix[r][right] for r in range(top, bottom + 1)]
        result += seg; right -= 1
        print(f"  xuống (cột {right + 1}): {seg}")
        if top <= bottom:
            seg = [matrix[bottom][c] for c in range(right, left - 1, -1)]
            result += seg; bottom -= 1
            print(f"  trái  (hàng {bottom + 1}): {seg}")
        else:
            print("  trái : bỏ qua (hết hàng)")
        if left <= right:
            seg = [matrix[r][left] for r in range(bottom, top - 1, -1)]
            result += seg; left += 1
            print(f"  lên   (cột {left - 1}): {seg}")
        else:
            print("  lên  : bỏ qua (hết cột)")
        layer += 1
    print("Kết quả:", result)
    return result

_ = trace_spiral([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]])

--- vòng 0: top=0 bottom=2 left=0 right=3
  phải  (hàng 0): [1, 2, 3, 4]
  xuống (cột 3): [8, 12]
  trái  (hàng 2): [11, 10, 9]
  lên   (cột 0): [5]
--- vòng 1: top=1 bottom=1 left=1 right=2
  phải  (hàng 1): [6, 7]
  xuống (cột 2): []
  trái : bỏ qua (hết hàng)
  lên   (cột 1): []
Kết quả: [1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]


Ở vòng 1, chỉ còn một hàng `[6, 7]`: bước "phải" lấy `6, 7`, bước "xuống" rỗng, rồi `top > bottom` nên bỏ bước "trái" — không lấy lại `7, 6`.

# 5. Cách 2: hướng đi + đánh dấu đã thăm

"Đi thẳng theo hướng hiện tại; nếu ô tiếp theo ra ngoài hoặc đã thăm thì **rẽ phải**."

Bốn hướng theo thứ tự rẽ phải: phải `(0, 1)`, xuống `(1, 0)`, trái `(0, -1)`, lên `(-1, 0)`. Rẽ phải = `d = (d + 1) % 4`.

In [7]:
def spiral_order_directions(matrix: List[List[int]]) -> List[int]:
    m, n = len(matrix), len(matrix[0])
    seen = [[False] * n for _ in range(m)]
    dirs = [(0, 1), (1, 0), (0, -1), (-1, 0)]
    r = c = d = 0
    result = []
    for _ in range(m * n):
        result.append(matrix[r][c])
        seen[r][c] = True
        nr, nc = r + dirs[d][0], c + dirs[d][1]
        if not (0 <= nr < m and 0 <= nc < n and not seen[nr][nc]):
            d = (d + 1) % 4                       # rẽ phải
            nr, nc = r + dirs[d][0], c + dirs[d][1]
        r, c = nr, nc
    return result

print(spiral_order_directions([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))
print(spiral_order_directions([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]))

[1, 2, 3, 6, 9, 8, 7, 4, 5]
[1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]


Cách này dễ viết đúng (ít trường hợp đặc biệt) nhưng tốn thêm `O(m · n)` bộ nhớ cho `seen`. Vòng lặp chạy **đúng `m · n` lần** nên không lo đi lố.

# 6. Phiên bản đúng format LeetCode

In [8]:
class Solution:
    def spiralOrder(self, matrix: List[List[int]]) -> List[int]:
        result = []
        top, bottom = 0, len(matrix) - 1
        left, right = 0, len(matrix[0]) - 1
        while top <= bottom and left <= right:
            for c in range(left, right + 1):
                result.append(matrix[top][c])
            top += 1
            for r in range(top, bottom + 1):
                result.append(matrix[r][right])
            right -= 1
            if top <= bottom:
                for c in range(right, left - 1, -1):
                    result.append(matrix[bottom][c])
                bottom -= 1
            if left <= right:
                for r in range(bottom, top - 1, -1):
                    result.append(matrix[r][left])
                left += 1
        return result

In [9]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<18} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.spiralOrder([[1, 2, 3], [4, 5, 6], [7, 8, 9]]), [1, 2, 3, 6, 9, 8, 7, 4, 5])
check("Ví dụ 2", sol.spiralOrder([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]), [1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7])
check("1 x 1", sol.spiralOrder([[7]]), [7])
check("Một hàng", sol.spiralOrder([[1, 2, 3]]), [1, 2, 3])
check("Một cột", sol.spiralOrder([[1], [2], [3]]), [1, 2, 3])
check("Hai cột", sol.spiralOrder([[1, 2], [3, 4], [5, 6]]), [1, 2, 4, 6, 5, 3])
check("4 x 4", sol.spiralOrder([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]]),
      [1, 2, 3, 4, 8, 12, 16, 15, 14, 13, 9, 5, 6, 7, 11, 10])

import random
random.seed(54)
for _ in range(2000):
    m, n = random.randint(1, 10), random.randint(1, 10)
    mat = [[random.randint(-100, 100) for _ in range(n)] for _ in range(m)]
    a = sol.spiralOrder(mat)
    assert a == spiral_order_directions(mat)
    assert sorted(a) == sorted(x for row in mat for x in row)    # đủ mọi phần tử, không thừa
print("2000 ma trận ngẫu nhiên: hai cách khớp nhau, lấy đủ mọi phần tử đúng một lần")

Ví dụ 1            PASS  [1, 2, 3, 6, 9, 8, 7, 4, 5]
Ví dụ 2            PASS  [1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]
1 x 1              PASS  [7]
Một hàng           PASS  [1, 2, 3]
Một cột            PASS  [1, 2, 3]
Hai cột            PASS  [1, 2, 4, 6, 5, 3]
4 x 4              PASS  [1, 2, 3, 4, 8, 12, 16, 15, 14, 13, 9, 5, 6, 7, 11, 10]
2000 ma trận ngẫu nhiên: hai cách khớp nhau, lấy đủ mọi phần tử đúng một lần


# 7. Vì sao đúng? Độ phức tạp?

**Bất biến:** ở đầu mỗi vòng `while`, phần **chưa duyệt** chính là hình chữ nhật `[top..bottom] × [left..right]`, và `result` chứa đúng các phần tử bên ngoài nó theo thứ tự xoắn ốc.

Mỗi bước trong vòng bóc đúng một cạnh của hình chữ nhật đó và co biên tương ứng. Hai câu `if` đảm bảo không bóc lại một hàng / cột đã bóc khi hình chữ nhật suy biến thành một đường.

```text
Time:  O(m · n)   — mỗi phần tử thêm đúng một lần
Space: O(1)       — ngoài mảng kết quả (cách 2: O(m · n) cho seen)
```

# 8. Những lỗi dễ mắc

## Lỗi 1: Thiếu kiểm tra ở bước "trái" và "lên"

In [10]:
def wrong_no_checks(matrix: List[List[int]]) -> List[int]:
    result = []
    top, bottom = 0, len(matrix) - 1
    left, right = 0, len(matrix[0]) - 1
    while top <= bottom and left <= right:
        for c in range(left, right + 1):
            result.append(matrix[top][c])
        top += 1
        for r in range(top, bottom + 1):
            result.append(matrix[r][right])
        right -= 1
        for c in range(right, left - 1, -1):      # thiếu if top <= bottom
            result.append(matrix[bottom][c])
        bottom -= 1
        for r in range(bottom, top - 1, -1):      # thiếu if left <= right
            result.append(matrix[r][left])
        left += 1
    return result

print("Sai :", wrong_no_checks([[1, 2, 3]]), "| Đúng:", sol.spiralOrder([[1, 2, 3]]))
print("Sai :", wrong_no_checks([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]))

Sai : [1, 2, 3, 2, 1] | Đúng: [1, 2, 3]
Sai : [1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7, 6]


## Lỗi 2: `range` ngược thiếu `- 1`

`range(right, left, -1)` **bỏ sót** cột `left`. Phải là `range(right, left - 1, -1)`.

## Lỗi 3: Lặp với điều kiện `while len(result) < m * n` nhưng vẫn thiếu kiểm tra

Điều kiện này giúp không lặp quá, nhưng **trong** một vòng vẫn có thể lấy thừa phần tử trước khi kiểm tra lại → kết quả dài hơn `m · n`.

## Lỗi 4: Nhầm `m` (số hàng) với `n` (số cột)

`len(matrix)` là số **hàng**, `len(matrix[0])` là số **cột**. Với ma trận không vuông, nhầm hai giá trị này gây `IndexError`.

# 9. Mẹo nhận diện pattern

**Mô phỏng trên ma trận:**

- Duyệt theo lớp / co biên: **#54**, **#59 Spiral Matrix II** (điền xoắn ốc), **#48 Rotate Image** (xoay theo lớp).
- Bảng hướng `dirs = [(0,1), (1,0), (0,-1), (-1,0)]` + `d = (d + 1) % 4`: robot, mê cung, **#885 Spiral Matrix III**, **#874 Walking Robot Simulation**.
- Mọi bài duyệt ma trận đều nên thử **ma trận 1 hàng, 1 cột, 1 × 1** trước khi nộp.

## Template (bốn biên)

```python
top, bottom, left, right = 0, m - 1, 0, n - 1
while top <= bottom and left <= right:
    phải trên hàng top;            top += 1
    xuống trên cột right;          right -= 1
    if top <= bottom: trái trên hàng bottom;  bottom -= 1
    if left <= right: lên trên cột left;      left += 1
```

# 10. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Duyệt xoắn ốc ma trận `3 × 5` có giá trị `1..15` (theo hàng). Ghi các biên sau mỗi vòng.

## Bài 2 — Điền chỗ trống

In [11]:
def exercise_2_spiral_order(matrix: List[List[int]]) -> List[int]:
    result = []
    top, bottom = 0, len(matrix) - 1
    left, right = 0, len(matrix[0]) - 1
    while top <= bottom and left <= right:
        # TODO 1: phải, co top
        # TODO 2: xuống, co right
        # TODO 3: nếu còn hàng: trái, co bottom
        # TODO 4: nếu còn cột: lên, co left
        pass
    return result


# assert exercise_2_spiral_order([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [1, 2, 3, 6, 9, 8, 7, 4, 5]
# assert exercise_2_spiral_order([[1, 2, 3]]) == [1, 2, 3]
# assert exercise_2_spiral_order([[1], [2], [3]]) == [1, 2, 3]

## Bài 3 — Biến thể: Spiral Matrix II (LeetCode #59)

Tạo ma trận `n × n` điền các số `1..n²` theo thứ tự xoắn ốc.

```python
generate_spiral(3) -> [[1, 2, 3], [8, 9, 4], [7, 6, 5]]
```

In [12]:
def exercise_3_generate_spiral(n: int) -> List[List[int]]:
    # TODO
    pass


# assert exercise_3_generate_spiral(3) == [[1, 2, 3], [8, 9, 4], [7, 6, 5]]
# assert exercise_3_generate_spiral(1) == [[1]]

## Bài 4 — Biến thể: duyệt theo đường chéo zigzag (LeetCode #498)

Duyệt ma trận theo các đường chéo, đổi chiều luân phiên.

```text
1 2 3
4 5 6    ->  [1, 2, 4, 7, 5, 3, 6, 8, 9]
7 8 9
```

Gợi ý nhanh: các ô trên cùng một đường chéo có cùng `r + c`. Gom theo `r + c`, rồi đảo các nhóm có chỉ số chẵn.

In [13]:
def exercise_4_diagonal_order(matrix: List[List[int]]) -> List[int]:
    # TODO
    pass


# assert exercise_4_diagonal_order([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [1, 2, 4, 7, 5, 3, 6, 8, 9]
# assert exercise_4_diagonal_order([[1, 2], [3, 4]]) == [1, 2, 3, 4]

## Bài 5 — Mini challenge: tọa độ thứ `k` không cần duyệt

Cho `m`, `n` và `k` (0-based). Trả về tọa độ `(r, c)` của phần tử thứ `k` trong thứ tự xoắn ốc, **không** duyệt từng ô (bỏ qua nguyên lớp nếu `k` nằm ở lớp trong).

```python
kth_in_spiral(3, 4, 0)  -> (0, 0)
kth_in_spiral(3, 4, 5)  -> (2, 3)       # thứ tự: 1,2,3,4,8,12 -> phần tử 12 ở (2,3)
kth_in_spiral(3, 4, 11) -> (1, 2)
```

Gợi ý nhanh: lớp ngoài của hình `h × w` có `2(h + w) - 4` ô (nếu `h, w >= 2`).

In [14]:
def exercise_5_kth_in_spiral(m: int, n: int, k: int):
    # TODO
    pass


# assert exercise_5_kth_in_spiral(3, 4, 0) == (0, 0)
# assert exercise_5_kth_in_spiral(3, 4, 5) == (2, 3)
# assert exercise_5_kth_in_spiral(3, 4, 11) == (1, 2)

# 11. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_spiral` sau khi tự làm để so.

**Bài 2**
- Mức 1: so với code mục 3.

**Bài 3**
- Mức 1: dùng cùng 4 biên, nhưng **gán** `mat[r][c] = num; num += 1` thay vì đọc.

**Bài 4**
- Mức 1: `groups = defaultdict(list)`; `groups[r + c].append(matrix[r][c])` (duyệt theo hàng).
- Mức 2: nhóm `s` chẵn đi từ dưới lên → đảo ngược nhóm đó.

**Bài 5**
- Mức 1: lặp qua các lớp: nếu `k >= số ô của lớp` thì `k -= số ô`, thu hẹp `h -= 2`, `w -= 2`, dời góc `(top, left) += 1`.
- Mức 2: trong lớp chứa `k`, xác định cạnh nào: trên (`w` ô), phải (`h - 1` ô), dưới (`w - 1` ô), trái (`h - 2` ô).
- Mức 3: cẩn thận lớp suy biến `h == 1` hoặc `w == 1`.

# 12. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [15]:
_ = trace_spiral([[c + 5 * r + 1 for c in range(5)] for r in range(3)])

--- vòng 0: top=0 bottom=2 left=0 right=4
  phải  (hàng 0): [1, 2, 3, 4, 5]
  xuống (cột 4): [10, 15]
  trái  (hàng 2): [14, 13, 12, 11]
  lên   (cột 0): [6]
--- vòng 1: top=1 bottom=1 left=1 right=3
  phải  (hàng 1): [7, 8, 9]
  xuống (cột 3): []
  trái : bỏ qua (hết hàng)
  lên   (cột 1): []
Kết quả: [1, 2, 3, 4, 5, 10, 15, 14, 13, 12, 11, 6, 7, 8, 9]


## Đáp án Bài 2

In [16]:
def answer_2_spiral_order(matrix: List[List[int]]) -> List[int]:
    return Solution().spiralOrder(matrix)

assert answer_2_spiral_order([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [1, 2, 3, 6, 9, 8, 7, 4, 5]
assert answer_2_spiral_order([[1, 2, 3]]) == [1, 2, 3]
assert answer_2_spiral_order([[1], [2], [3]]) == [1, 2, 3]
print("Bài 2 PASS (giống lời giải chính ở mục 6)")

Bài 2 PASS (giống lời giải chính ở mục 6)


## Đáp án Bài 3

In [17]:
def answer_3_generate_spiral(n: int) -> List[List[int]]:
    mat = [[0] * n for _ in range(n)]
    top, bottom, left, right = 0, n - 1, 0, n - 1
    num = 1
    while top <= bottom and left <= right:
        for c in range(left, right + 1):
            mat[top][c] = num; num += 1
        top += 1
        for r in range(top, bottom + 1):
            mat[r][right] = num; num += 1
        right -= 1
        if top <= bottom:
            for c in range(right, left - 1, -1):
                mat[bottom][c] = num; num += 1
            bottom -= 1
        if left <= right:
            for r in range(bottom, top - 1, -1):
                mat[r][left] = num; num += 1
            left += 1
    return mat

assert answer_3_generate_spiral(3) == [[1, 2, 3], [8, 9, 4], [7, 6, 5]]
assert answer_3_generate_spiral(1) == [[1]]
for n in range(1, 8):
    assert sol.spiralOrder(answer_3_generate_spiral(n)) == list(range(1, n * n + 1))
print("Bài 3 PASS (đọc xoắn ốc ma trận vừa tạo ra đúng 1..n²)")

Bài 3 PASS (đọc xoắn ốc ma trận vừa tạo ra đúng 1..n²)


## Đáp án Bài 4

In [18]:
from collections import defaultdict

def answer_4_diagonal_order(matrix: List[List[int]]) -> List[int]:
    groups = defaultdict(list)
    for r, row in enumerate(matrix):
        for c, v in enumerate(row):
            groups[r + c].append(v)
    result = []
    for s in range(len(matrix) + len(matrix[0]) - 1):
        diag = groups[s]
        result.extend(reversed(diag) if s % 2 == 0 else diag)
    return result

assert answer_4_diagonal_order([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [1, 2, 4, 7, 5, 3, 6, 8, 9]
assert answer_4_diagonal_order([[1, 2], [3, 4]]) == [1, 2, 3, 4]
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [19]:
def answer_5_kth_in_spiral(m: int, n: int, k: int):
    top, left, h, w = 0, 0, m, n
    while True:
        if h == 1:
            return (top, left + k)
        if w == 1:
            return (top + k, left)
        ring = 2 * (h + w) - 4
        if k >= ring:
            k -= ring
            top += 1; left += 1; h -= 2; w -= 2
            continue
        if k < w:                                   # cạnh trên
            return (top, left + k)
        k -= w
        if k < h - 1:                               # cạnh phải
            return (top + 1 + k, left + w - 1)
        k -= h - 1
        if k < w - 1:                               # cạnh dưới
            return (top + h - 1, left + w - 2 - k)
        k -= w - 1
        return (top + h - 2 - k, left)              # cạnh trái

assert answer_5_kth_in_spiral(3, 4, 0) == (0, 0)
assert answer_5_kth_in_spiral(3, 4, 5) == (2, 3)
assert answer_5_kth_in_spiral(3, 4, 11) == (1, 2)
for m in range(1, 9):
    for n in range(1, 9):
        coords = sol.spiralOrder([[(r, c) for c in range(n)] for r in range(m)])
        assert all(answer_5_kth_in_spiral(m, n, k) == coords[k] for k in range(m * n)), (m, n)
print("Bài 5 PASS (khớp mọi k với mọi kích thước 1..8 x 1..8)")

Bài 5 PASS (khớp mọi k với mọi kích thước 1..8 x 1..8)


Mẹo kiểm tra trong đáp án: tạo ma trận mà **mỗi ô chứa chính tọa độ của nó**, rồi đọc xoắn ốc → được danh sách tọa độ theo thứ tự.

# 13. Tổng kết

> Giữ bốn biên của phần chưa duyệt. Mỗi vòng: **phải → xuống → (nếu còn hàng) trái → (nếu còn cột) lên**, co biên sau mỗi cạnh. Hai câu `if` là chìa khóa cho ma trận một hàng / một cột.

---

## Checklist trước khi nộp

- [ ] Điều kiện vòng `top <= bottom and left <= right`?
- [ ] `if top <= bottom` trước bước trái, `if left <= right` trước bước lên?
- [ ] `range` ngược dùng `left - 1` / `top - 1`?
- [ ] Đã thử `1 × 1`, một hàng, một cột, ma trận không vuông?